# 02 — Main analysis

**Research question:** As Australia's material conditions improved, did perceived social support and emotional well-being deteriorate relative to comparable countries?

**Primary outcomes:** real PPP-adjusted household income per person (USD), employment rate (percentage points), lack of social support (percentage points), and negative affect (percentage points).  
**Method:** descriptive common-endpoint comparisons and same-year gaps. Countries must report both exact endpoints; lower-is-better measures are sign-oriented before ranking.  
**Success criterion:** a conclusion must be stable across the pre-specified English-speaking peer group, the broad supplied-country reference set, normal-value-only data, and leave-one-peer-out checks. This is not a causal design.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.oecd_audit import INDICATOR_SPECS, load_clean

ENGLISH_SPEAKING_PEERS = ['CAN', 'NZL', 'GBR', 'USA']
PRIMARY_CODES = ['1_1', '2_1', '2_7', '2_2', '1_2']
MATERIAL_SOCIAL_SPECS = [
    ('1_1', 2010, 2024, '2010–2024'),
    ('2_1', 2010, 2024, '2010–2024'),
    ('7_1_DEP', 2010, 2024, '2008–10 to 2023–25 pooled windows'),
    ('11_2', 2010, 2024, '2008–10 to 2023–25 pooled windows'),
]
TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'

def endpoint_changes(data, code, references, start=2010, end=2024, normal_only=False):
    subset = data.loc[data.indicator_code.eq(code) & data.country_code.isin(['AUS', *references]) & data.year.isin([start, end])].copy()
    if normal_only:
        subset = subset.loc[subset.status_code.eq('A')]
    subset = subset.sort_values('year').drop_duplicates(['country_code', 'independent_period'], keep='last')
    wide = subset.pivot(index='country_code', columns='year', values='value').reindex(columns=[start, end]).dropna().reset_index()
    wide = wide.rename(columns={start: 'start_value', end: 'end_value'})
    wide['absolute_change'] = wide.end_value - wide.start_value
    wide['oriented_change'] = wide.absolute_change * (1 if INDICATOR_SPECS[code].direction == 'higher' else -1)
    return wide


def material_social_endpoints(data):
    """Build the four-outcome table using only exact common endpoints."""
    all_references = sorted(set(data.country_code) - {'AUS'})
    rows = []
    for code, start, end, period_label in MATERIAL_SOCIAL_SPECS:
        changes = endpoint_changes(data, code, all_references, start, end)
        if 'AUS' not in changes.country_code.values:
            raise ValueError(f'Australia lacks a required endpoint for {code}.')
        australia = changes.loc[changes.country_code.eq('AUS')].iloc[0]
        comparators = changes.loc[changes.country_code.ne('AUS')]
        comparator_count = int(len(comparators))
        australia_rank = changes.oriented_change.rank(ascending=False, method='average').loc[australia.name]
        total_country_count = len(changes)
        rows.append({
            'indicator_code': code,
            'indicator': data.loc[data.indicator_code.eq(code), 'indicator'].iloc[0],
            'unit': data.loc[data.indicator_code.eq(code), 'unit'].iloc[0],
            'comparison_period': period_label,
            'start_year_displayed': start,
            'end_year_displayed': end,
            'australia_start_value': australia.start_value,
            'australia_end_value': australia.end_value,
            'australia_native_change': australia.absolute_change,
            'comparator_median_native_change': comparators.absolute_change.median(),
            'australia_minus_comparator_median_oriented': australia.oriented_change - comparators.oriented_change.median(),
            'australia_favourable_percentile': 100 * (total_country_count - australia_rank) / (total_country_count - 1),
            'eligible_comparator_country_count': comparator_count,
        })
    return pd.DataFrame(rows)

def comparative_scorecard(data, codes=PRIMARY_CODES, references=ENGLISH_SPEAKING_PEERS, start=2010, end=2024, normal_only=False):
    rows = []
    for code in codes:
        changes = endpoint_changes(data, code, references, start, end, normal_only)
        aus, peers = changes.loc[changes.country_code.eq('AUS')], changes.loc[changes.country_code.ne('AUS')]
        if aus.empty or peers.empty: continue
        au = aus.iloc[0]; rank = changes.oriented_change.rank(ascending=False, method='average').loc[aus.index[0]]; n = len(changes)
        rows.append({'indicator_code': code, 'indicator': data.loc[data.indicator_code.eq(code), 'indicator'].iloc[0], 'start_year': start, 'end_year': end, 'australia_start_value': au.start_value, 'australia_end_value': au.end_value, 'australia_absolute_change': au.absolute_change, 'australia_oriented_change': au.oriented_change, 'reference_country_count': len(peers), 'reference_median_oriented_change': peers.oriented_change.median(), 'australia_minus_reference_median': au.oriented_change - peers.oriented_change.median(), 'australia_change_percentile': 100 if n == 1 else 100 * (n-rank)/(n-1), 'normal_values_only': normal_only})
    return pd.DataFrame(rows)

def relative_trend(data, code, references, start=2010, end=2024):
    subset = data.loc[data.indicator_code.eq(code) & data.country_code.isin(['AUS', *references]) & data.year.between(start, end)].sort_values('year').drop_duplicates(['country_code', 'independent_period'], keep='last')
    aus = subset.loc[subset.country_code.eq('AUS'), ['year', 'value']].rename(columns={'value': 'australia_value'})
    peer = subset.loc[subset.country_code.ne('AUS')].groupby('year', as_index=False).agg(reference_median=('value', 'median'), reference_country_count=('country_code', 'nunique'))
    out = aus.merge(peer, on='year'); out['oriented_gap'] = (out.australia_value-out.reference_median) * (1 if INDICATOR_SPECS[code].direction == 'higher' else -1)
    return out

def leave_one_out(data, codes=PRIMARY_CODES):
    results = []
    for omitted in ENGLISH_SPEAKING_PEERS:
        table = comparative_scorecard(data, codes, [p for p in ENGLISH_SPEAKING_PEERS if p != omitted])
        table.insert(0, 'omitted_peer', omitted); results.append(table)
    return pd.concat(results, ignore_index=True)

df = load_clean()
df.shape

## Pre-specified descriptive baseline

This is a small country-level panel, not an experiment. We therefore report transparent effect sizes and coverage rather than p-values that would overstate precision. The primary sensitivity group—Canada, New Zealand, the United Kingdom and the United States—is interpretable but not uniquely correct; all supplied countries are the breadth check.

Method 2 starts with a single four-outcome common-endpoint workflow. A country enters an outcome-specific comparison only when it reports both of Australia's exact displayed endpoints: 2010 and 2024 for income/employment, and the displayed rows representing the 2008–10 and 2023–25 pooled social windows. The table retains native-unit changes for interpretation, then adds a direction-oriented Australia-minus-comparator-median gap and favourable percentile for cross-outcome comparison. Uncertainty and robustness are subsequent Method 2 steps.

In [ ]:
# Method 2, steps 1–3: one four-outcome workflow with exact common endpoints.
primary_endpoints = material_social_endpoints(df)

# Correctness checks: all primary outcomes are present, every reported
# Australian endpoint comes directly from the audited tidy data, and only
# countries with both exact displayed endpoints are retained as comparators.
assert primary_endpoints.indicator_code.tolist() == [spec[0] for spec in MATERIAL_SOCIAL_SPECS]
assert primary_endpoints[['australia_start_value', 'australia_end_value', 'australia_native_change']].notna().all().all()
assert primary_endpoints[['comparator_median_native_change', 'australia_minus_comparator_median_oriented', 'australia_favourable_percentile']].notna().all().all()
assert primary_endpoints.eligible_comparator_country_count.tolist() == [31, 43, 46, 46]
assert primary_endpoints.australia_favourable_percentile.between(0, 100).all()
expected_australia_endpoints = {
    '1_1': (44625.0, 50629.0), '2_1': (75.444, 80.262),
    '7_1_DEP': (4.926543, 10.043193), '11_2': (12.244020, 14.853043),
}
for row in primary_endpoints.itertuples(index=False):
    expected_start, expected_end = expected_australia_endpoints[row.indicator_code]
    assert abs(row.australia_start_value - expected_start) < 1e-5
    assert abs(row.australia_end_value - expected_end) < 1e-5
    common = endpoint_changes(df, row.indicator_code, sorted(set(df.country_code) - {'AUS'}), row.start_year_displayed, row.end_year_displayed)
    assert len(common) == row.eligible_comparator_country_count + 1
    comparators = common.loc[common.country_code.ne('AUS')]
    sign = 1 if INDICATOR_SPECS[row.indicator_code].direction == 'higher' else -1
    expected_gap = sign * (row.australia_native_change - comparators.absolute_change.median())
    assert abs(row.australia_minus_comparator_median_oriented - expected_gap) < 1e-10
    expected_rank = common.oriented_change.rank(ascending=False, method='average').loc[common.country_code.eq('AUS')].iloc[0]
    expected_percentile = 100 * (len(common) - expected_rank) / (len(common) - 1)
    assert abs(row.australia_favourable_percentile - expected_percentile) < 1e-10

display(primary_endpoints)

# Existing economic exploration outputs retained for later sensitivity work.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
english_peers = comparative_scorecard(df)
all_countries = comparative_scorecard(df, references=sorted(set(df.country_code) - {'AUS'}))
normal_only = comparative_scorecard(df, references=sorted(set(df.country_code) - {'AUS'}), normal_only=True)
inequality = comparative_scorecard(df, codes=['1_2'], references=sorted(set(df.country_code) - {'AUS'}), start=2012, end=2020)
english_peers.to_csv(TABLE_DIR / 'economic_change_scorecard_english_peers.csv', index=False)
all_countries.to_csv(TABLE_DIR / 'economic_change_scorecard_all_countries.csv', index=False)
normal_only.to_csv(TABLE_DIR / 'economic_change_scorecard_normal_values.csv', index=False)
inequality.to_csv(TABLE_DIR / 'income_inequality_change_scorecard.csv', index=False)
leave_one_out(df).to_csv(TABLE_DIR / 'economic_leave_one_out.csv', index=False)
for code, name in [('1_1', 'income'), ('2_1', 'employment'), ('2_7', 'long_hours')]:
    relative_trend(df, code, ENGLISH_SPEAKING_PEERS).to_csv(TABLE_DIR / f'{name}_relative_trend_english_peers.csv', index=False)
display(english_peers)
display(all_countries)

## Relative trends and robustness

The trend table reports only same-year comparisons, including the number of reporting reference countries. A positive oriented gap is favourable for Australia. Leave-one-out results identify conclusions driven by a single peer; normal-value-only results check sensitivity to OECD status flags.

Interpretation is deliberately constrained: income and long-hours improvement are clear within Australia, but income and employment change are not consistently stronger than the broad reference set. The data support a nuanced material-progress story, not a claim that gains were broadly shared—distributional evidence is sparse and S80/S20 did not improve between 2012 and 2020.